<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m7_04_classification_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M7.04 — Whom should the bank call?

In the lab we saw that accuracy says little when one class is rare, and
met the confusion matrix, precision, recall and the curves obtained by
moving the decision threshold. In this workbook you evaluate a model
that tells a bank **which clients to phone**. You will choose a
threshold from what a call costs and what a sale brings, catch a feature
that makes the model look far better than it can be, and see that a
model can rank clients well and still give probabilities you should not
believe.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "accuracy_gain": "55edad7c6cf5e696",
    "balanced_ranks_better": "9390298f3fb0c5b1",
    "believable_probabilities": "a116c9ed46d62077",
    "best_strategy": "cebbedfeaa3e289f",
    "break_even_probability": "14be4b45f18e0d8c",
    "duration_known_before_calling": "9390298f3fb0c5b1",
    "precision_example": "1e9d7c27c8bbc8dd",
    "precision_when_threshold_lowered": "bcf47e35fc5fe9b7",
    "recall_example": "44896b09365746b5",
    "recall_when_threshold_lowered": "69421502d03aee93",
    "usable_model": "3976db882b5c8bf8"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

A Portuguese bank phoned its clients to offer a savings product. Each
row is **one call**, with what the bank knew about the client (age, job,
marital status, education, account balance, existing loans), about the
campaign (month, number of calls already made to this client, result of
the previous campaign) and about the call itself. `subscribed` says
whether the client accepted.

The bank wants a model that says, **before picking up the phone**,
which clients are worth calling.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

calls = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/bank_marketing.csv"
)
target = calls["subscribed"] == "yes"
print(f"{len(calls)} calls, share of clients who subscribed: {target.mean():.3f}")
calls.head()

We set the column `duration` (length of the call in seconds) aside for
now; section 5 comes back to it. The model is a logistic regression on
one-hot encoded and scaled features, as in module 1.

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_model(**logistic_options):
    preprocessor = make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_exclude="number")),
        (StandardScaler(), make_column_selector(dtype_include="number")),
    )
    return make_pipeline(preprocessor, LogisticRegression(max_iter=2000, **logistic_options))


data = calls.drop(columns=["subscribed", "duration"])
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0, stratify=target
)
model = make_model().fit(data_train, target_train)

## 2. Accuracy, and why it is not enough

🔮 **Predict first.** About 89% of the clients say no. The accuracy of
our model on the test calls will be **"below 0.80"**, **"around 0.90"**
or **"above 0.97"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["below 0.80", "around 0.90", "above 0.97"])

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

always_no = DummyClassifier(strategy="most_frequent").fit(data_train, target_train)
accuracy = {
    "always answer no": accuracy_score(target_test, always_no.predict(data_test)),
    "logistic regression": accuracy_score(target_test, model.predict(data_test)),
}
pd.Series(accuracy, name="accuracy").round(3).to_frame()

✏️ How much accuracy did the model gain over never calling anybody?
Write `"almost nothing"` or `"a lot"`.

In [ ]:
accuracy_gain = ____
check("accuracy_gain", accuracy_gain)

Accuracy counts all errors alike and is dominated by the many clients
who say no. It cannot tell us whether the model finds the clients who
say **yes**, which is the only thing the bank cares about.

The **confusion matrix** separates the four possible outcomes.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(4.5, 3.5))
_ = ConfusionMatrixDisplay.from_estimator(
    model, data_test, target_test, display_labels=["said no", "said yes"], colorbar=False, ax=ax
)
_ = ax.set(xlabel="the model predicts", ylabel="the client really")

Two questions can be asked about the clients who say yes:

- **recall**: of all the clients who would say yes, which share does the
  model find?
- **precision**: of all the clients the model tells us to call, which
  share says yes?

✏️ Practise on round numbers. Among 1000 clients, 100 would say yes. A
model tells the bank to call 25 clients, and 20 of them say yes. What
are its recall and its precision?

In [ ]:
recall_example = ____
precision_example = ____
check("recall_example", recall_example, hint="Buyers found, divided by all the buyers.")
check("precision_example", precision_example, hint="Buyers found, divided by the clients called.")

In [ ]:
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score

predicted = model.predict(data_test)
pd.Series(
    {
        "accuracy": accuracy_score(target_test, predicted),
        "balanced accuracy": balanced_accuracy_score(target_test, predicted),
        "precision": precision_score(target_test, predicted),
        "recall": recall_score(target_test, predicted),
    },
    name="logistic regression",
).round(3).to_frame()

Our real model resembles the example: when it says "call", it is often
right (precision), but it finds only a small part of the buyers
(recall). The **balanced accuracy** is the average of the recall on each
class; 0.5 is the level of a model that knows nothing.

## 3. Moving the threshold

The model computes a probability for each client and says "call" when
it is above **0.5**. Nothing forces us to keep 0.5. Here is what happens
with other thresholds.

In [ ]:
probability = model.predict_proba(data_test)[:, 1]
rows = {}
for threshold in [0.5, 0.3, 0.2, 0.1, 0.05]:
    call = probability > threshold
    rows[threshold] = {
        "clients called": int(call.sum()),
        "precision": precision_score(target_test, call),
        "recall": recall_score(target_test, call),
    }
by_threshold = pd.DataFrame(rows).T.rename_axis("threshold").astype({"clients called": int})
by_threshold.round(3)

✏️ When the threshold is **lowered**, what happens to the recall and to
the precision? Write `"goes up"` or `"goes down"` for each.

In [ ]:
recall_when_threshold_lowered = ____
precision_when_threshold_lowered = ____
check("recall_when_threshold_lowered", recall_when_threshold_lowered)
check("precision_when_threshold_lowered", precision_when_threshold_lowered)

The model has not changed: one model gives a whole family of possible
decisions. The **precision-recall curve** shows all of them at once.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, ax = plt.subplots(figsize=(5.5, 4))
PrecisionRecallDisplay.from_predictions(target_test, probability, ax=ax, name="logistic regression")
ax.axhline(target_test.mean(), linestyle="--", color="gray", label="calling at random")
_ = ax.legend(loc="upper right")

## 4. ➕ Going further: the threshold that earns the most

Which point of the curve should the bank choose? That is not a question
of statistics but of money. Suppose that

- a call costs the bank **€5** (the time of an employee);
- a client who subscribes brings **€50**.

✏️ A client has a probability `p` of saying yes. Calling him brings
`50 × p` euros on average and costs 5. Above which probability is the
call worth making?

In [ ]:
break_even_probability = ____
check("break_even_probability", break_even_probability, hint="Solve 50 × p = 5.")

So the sensible threshold is far below 0.5. scikit-learn can search for
it directly: `TunedThresholdClassifierCV` tries many thresholds by
cross-validation **on the training data** and keeps the one that
maximises a score of our choice, here the money earned.

In [ ]:
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import TunedThresholdClassifierCV


def money_earned(real, called):
    _, wasted_calls, _, sales = confusion_matrix(real, called).ravel()
    return 45 * sales - 5 * wasted_calls


tuned = TunedThresholdClassifierCV(make_model(), scoring=make_scorer(money_earned), cv=5, random_state=0)
tuned.fit(data_train, target_train)
print(f"threshold chosen: {tuned.best_threshold_:.2f}")

strategies = {
    "call nobody": np.zeros(len(target_test), dtype=bool),
    "call everybody": np.ones(len(target_test), dtype=bool),
    "model, threshold 0.5": model.predict(data_test),
    "model, tuned threshold": tuned.predict(data_test),
}
earnings = pd.DataFrame(
    {
        name: {"clients called": int(called.sum()), "money earned (€)": money_earned(target_test, called)}
        for name, called in strategies.items()
    }
).T
earnings

✏️ Which strategy earns the most on the test clients? Write its name as
it appears in the table.

In [ ]:
best_strategy = ____
check("best_strategy", best_strategy)

With the default threshold, the model was hardly better than calling
everybody. With a threshold that matches the costs, **the same model**
earns more than twice as much. Of all the settings of a classifier, the
threshold is the cheapest to change and often the one that matters most.
scikit-learn's example
[Post-tuning the decision threshold for cost-sensitive learning](https://scikit-learn.org/1.8/auto_examples/model_selection/plot_cost_sensitive_learning.html)
develops this.

🧠 **Think.** The threshold found by cross-validation is close to the
0.1 you computed by hand. What assumption about the model's
probabilities did your computation make? (Section 6 tests it.)

## 5. ➕ Going further: a feature that is too good

We left `duration` out. A colleague adds it back and reports a much
better model. We compare with the **ROC AUC**, which measures how well
the model ranks buyers above non-buyers (0.5: random order, 1: perfect).

In [ ]:
from sklearn.metrics import roc_auc_score

with_duration = calls.drop(columns="subscribed")
duration_train, duration_test = with_duration.loc[data_train.index], with_duration.loc[data_test.index]
leaky_model = make_model().fit(duration_train, target_train)
auc = {
    "without duration": roc_auc_score(target_test, probability),
    "with duration": roc_auc_score(target_test, leaky_model.predict_proba(duration_test)[:, 1]),
}
print(pd.Series(auc, name="ROC AUC").round(3).to_string())
calls.groupby("subscribed")["duration"].median().rename("median length of the call (seconds)").to_frame()

Calls that end with a subscription are much longer, of course: the
client stays on the line to give his details, and a client who hangs up
after ten seconds has said no.

✏️ The model must decide **whom to call**. At that moment, does the bank
know the duration of the call? Write `"yes"` or `"no"`. Which of the two
models can really be used: `"with duration"` or `"without duration"`?

In [ ]:
duration_known_before_calling = ____
usable_model = ____
check("duration_known_before_calling", duration_known_before_calling)
check("usable_model", usable_model)

This is **target leakage**: a feature that exists in the historical file
but will not exist when the model is used, because it is a consequence
of the answer. No metric and no cross-validation can detect it: the
leaky model really is better on every test set made from this file. Only
the question "when is this column filled in?" reveals it.

The original description of the dataset warns about this column for
exactly that reason.

## 6. ➕ Going further: can the probabilities be believed?

A popular remedy for rare classes is `class_weight="balanced"`: errors
on the rare class count more during training. We train that variant and
compare what the two models say.

In [ ]:
balanced_model = make_model(class_weight="balanced").fit(data_train, target_train)
balanced_probability = balanced_model.predict_proba(data_test)[:, 1]
comparison = pd.DataFrame(
    {
        "plain": {
            "ROC AUC": roc_auc_score(target_test, probability),
            "recall at threshold 0.5": recall_score(target_test, probability > 0.5),
            "mean predicted probability": probability.mean(),
        },
        "balanced": {
            "ROC AUC": roc_auc_score(target_test, balanced_probability),
            "recall at threshold 0.5": recall_score(target_test, balanced_probability > 0.5),
            "mean predicted probability": balanced_probability.mean(),
        },
    }
)
print(f"real share of buyers among the test clients: {target_test.mean():.3f}")
comparison.round(3)

In [ ]:
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(5.5, 4.5))
CalibrationDisplay.from_predictions(target_test, probability, n_bins=8, strategy="quantile", name="plain", ax=ax)
CalibrationDisplay.from_predictions(
    target_test, balanced_probability, n_bins=8, strategy="quantile", name="balanced", ax=ax
)
_ = ax.set(xlabel="probability announced by the model", ylabel="share of clients who really said yes")

On this plot, a model whose probabilities can be believed lies on the
diagonal: among the clients it gives 30%, 30% say yes.

✏️ Did the balanced model get better at **ranking** the clients (ROC
AUC)? Write `"yes"` or `"no"`. And which model announces probabilities
that match reality: `"plain"` or `"balanced"`?

In [ ]:
balanced_ranks_better = ____
believable_probabilities = ____
check("balanced_ranks_better", balanced_ranks_better)
check("believable_probabilities", believable_probabilities)

The balanced model orders the clients exactly as well as the plain one.
What changed is the **scale** of its probabilities: it announces 41% on
average when 11% of the clients say yes. Its higher recall at 0.5 is
only the effect of a moved threshold, the one you moved by hand in
section 3.

This matters for section 4: the rule "call if the probability is above
0.1" assumes that 0.1 means 10%. With the balanced model, that rule
would have the bank call almost everybody.

🧠 **Think.**

1. Two colleagues argue: one says the balanced model is better because
   its recall is 0.64 against 0.19; the other says the two models are
   the same. Who is right, and what would you measure to settle it?
2. A model for a rare disease reaches an accuracy of 99%. Which three
   numbers do you ask for before saying anything?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - why an accuracy of 0.90 means almost nothing here, and what
     precision and recall add;
   - the threshold: what moves when it is lowered, and how the costs of
     a call and of a sale fix it;
   - why `duration` cannot be used, although it improves every score;
   - the plain and the balanced model: same ranking, different
     probabilities.